# TinyLlama-1.1B LoRA — Baseline (Notebook 1 of 2)

Trains the fixed-hyperparameter LoRA baseline, logs per-epoch metrics, evaluates once on test, and writes `shared_config.json` for the companion Optuna notebook.

In [ ]:
%pip install transformers datasets accelerate peft scikit-learn pandas matplotlib optuna
%pip uninstall -y torchao


In [ ]:
import os, re, time, json, math, random, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import optuna

from datasets import Dataset

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_squared_log_error
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.trainer_utils import get_last_checkpoint

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel
)


## Cell — Reproducibility & GPU check

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is strongly recommended for fine-tuning TinyLlama-1.1B."
    )

print("GPU:", torch.cuda.get_device_name(0))
gpu_props = torch.cuda.get_device_properties(0)
print("GPU memory (GB):", round(gpu_props.total_memory / (1024**3), 2))


## Cell — Configuration

In [ ]:
# ------------------------------------------------------------------
# CHOOSE SERIALIZATION HERE: "exact" | "human_readable" | "prompt_style" | "json"
# ------------------------------------------------------------------
SERIALIZATION = "prompt_style"   # "exact" | "human_readable" | "prompt_style" | "json"

DATA_DIR = "/kaggle/input/datasets/tamislam/llm-serialisation-dataset-final/kickstarter_serializations"
TRAIN_FILE = f"{DATA_DIR}/{SERIALIZATION}/kickstarter_llm_train.csv"
TEST_FILE  = f"{DATA_DIR}/{SERIALIZATION}/kickstarter_llm_test.csv"

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

BASELINE_OUTPUT_DIR = f"./tinyllama_1_1b_{SERIALIZATION}_baseline_lora"   # Trainer checkpoints (for resume)
BASELINE_MODEL_DIR  = f"./baseline_model"                                # final adapter+tokenizer for handoff/XAI
EPOCH_METRICS_CSV   = "./baseline_epoch_metrics.csv"
FINAL_METRICS_CSV   = "./baseline_final_metrics.csv"
SHARED_CONFIG_PATH  = "./shared_config.json"

MAX_LENGTH = 512
MAX_NEW_TOKENS = 12
GENERATION_BATCH_SIZE = 32

SYSTEM_PROMPT = (
    "Predict the final Kickstarter funding value as log1p(USD) "
    "from the provided campaign features. "
    "Output only the numeric value."
)

# Baseline (fixed) hyperparameters -- same as the Qwen baseline notebook
BASELINE_PARAMS = dict(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    learning_rate=1e-4,
    num_train_epochs=4,
    train_batch_size=8,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)

VAL_FRACTION = 0.15   # carved out of the training file only; test file stays untouched

print("Serialization:", SERIALIZATION)
print("Train file:", TRAIN_FILE)
print("Test file:", TEST_FILE)


## Cell — Load data & carve out validation split from train (test stays held out)

In [ ]:
train_full_df = pd.read_csv(TRAIN_FILE, keep_default_na=False)
test_df = pd.read_csv(TEST_FILE, keep_default_na=False)

required_columns = {"id", "text", "target", "target_usd"}
assert required_columns.issubset(train_full_df.columns)
assert required_columns.issubset(test_df.columns)

for col in ["target", "target_usd"]:
    train_full_df[col] = pd.to_numeric(train_full_df[col], errors="raise")
    test_df[col] = pd.to_numeric(test_df[col], errors="raise")

overlap = set(train_full_df["id"]).intersection(set(test_df["id"]))
assert len(overlap) == 0, f"Train/test ID overlap: {len(overlap)}"

# Validation split carved from train only -- test is never touched until final eval.
# This exact split (by id) is written to shared_config.json so the Optuna notebook
# can rebuild the IDENTICAL train/val partition instead of re-sampling.
val_size = int(len(train_full_df) * VAL_FRACTION)
shuffled = train_full_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_df = shuffled.iloc[:val_size].reset_index(drop=True)
train_df = shuffled.iloc[val_size:].reset_index(drop=True)

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

fallback_log_target = float(train_full_df["target"].median())
max_reasonable_log_target = float(train_full_df["target"].max() + 2.0)
print("Fallback log target (training median):", fallback_log_target)
print("Max accepted generated log target:", max_reasonable_log_target)


## Cell — Tokenizer & shared helper functions (masked SFT examples, collator, parsing, metrics)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_bf16_supported():
    TRAIN_DTYPE = torch.bfloat16
    USE_BF16, USE_FP16 = True, False
else:
    TRAIN_DTYPE = torch.float16
    USE_BF16, USE_FP16 = False, True

print("Training dtype:", TRAIN_DTYPE, "| bf16:", USE_BF16, "| fp16:", USE_FP16)


def build_training_example(example):
    target_text = f"{float(example['target']):.4f}"
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(example["text"])},
    ]
    full_messages = prompt_messages + [{"role": "assistant", "content": target_text}]

    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)["input_ids"]
    full = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]
    labels = input_ids.copy()

    prompt_length = min(len(prompt_ids), len(labels))
    labels[:prompt_length] = [-100] * prompt_length

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


class CausalLMDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        batch_input_ids, batch_attention_mask, batch_labels = [], [], []
        for item in features:
            pad_len = max_len - len(item["input_ids"])
            batch_input_ids.append(item["input_ids"] + [self.tokenizer.pad_token_id] * pad_len)
            batch_attention_mask.append(item["attention_mask"] + [0] * pad_len)
            batch_labels.append(item["labels"] + [-100] * pad_len)
        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }


data_collator = CausalLMDataCollator(tokenizer)

NUMBER_PATTERN = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?")


def parse_generated_log_target(text):
    match = NUMBER_PATTERN.search(str(text))
    if match is None:
        return np.nan
    try:
        value = float(match.group(0))
    except Exception:
        return np.nan
    if not np.isfinite(value) or value < 0 or value > max_reasonable_log_target:
        return np.nan
    return value


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_regression(y_true_log, pred_log, actual_usd, pred_usd):
    return {
        "MAE_log": mean_absolute_error(y_true_log, pred_log),
        "MSE_log": mean_squared_error(y_true_log, pred_log),
        "RMSE_log": rmse(y_true_log, pred_log),
        "R2_log": r2_score(y_true_log, pred_log),
        "MAE_USD": mean_absolute_error(actual_usd, pred_usd),
        "MSE_USD": mean_squared_error(actual_usd, pred_usd),
        "RMSE_USD": rmse(actual_usd, pred_usd),
        "R2_USD": r2_score(actual_usd, pred_usd),
        "RMSLE": np.sqrt(mean_squared_log_error(actual_usd, pred_usd)),
    }


def build_inference_prompt(feature_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(feature_text)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_predictions(model, eval_df, batch_size=GENERATION_BATCH_SIZE):
    """Generate log_target predictions for a dataframe with a 'text' column.
    Returns (pred_log_array, valid_mask, generated_texts, generation_time_seconds)."""
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    device = next(model.parameters()).device

    prompts = [build_inference_prompt(x) for x in eval_df["text"]]
    generated_texts = []
    start_time = time.perf_counter()

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=MAX_LENGTH)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        prompt_width = inputs["input_ids"].shape[1]
        new_tokens = generated_ids[:, prompt_width:]
        batch_text = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        generated_texts.extend(batch_text)

    generation_time = time.perf_counter() - start_time

    pred_log_raw = np.array([parse_generated_log_target(t) for t in generated_texts], dtype=float)
    valid_mask = np.isfinite(pred_log_raw)
    pred_log = pred_log_raw.copy()
    pred_log[~valid_mask] = fallback_log_target

    tokenizer.padding_side = "right"
    model.config.use_cache = False

    return pred_log, valid_mask, generated_texts, generation_time


## Cell — LoRA model builder & tokenized dataset helper

In [ ]:
def make_model_with_lora(params):
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=TRAIN_DTYPE)
    model.config.use_cache = False

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=params["r"],
        lora_alpha=params["lora_alpha"],
        lora_dropout=params["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    return model


def tokenize_split(df):
    hf_ds = Dataset.from_pandas(df[["text", "target"]], preserve_index=False)
    return hf_ds.map(build_training_example, remove_columns=hf_ds.column_names)


## Cell — Per-epoch full-metric eval callback (resume-safe)

In [ ]:
class EpochEvalCallback(TrainerCallback):
    """Runs the full 9-metric generative eval at the end of every epoch and
    incrementally appends the result to `csv_path`, so progress survives a
    Kaggle session crash / restart (existing rows for completed epochs are kept)."""

    def __init__(self, eval_df, csv_path, extra_cols=None):
        self.eval_df = eval_df
        self.csv_path = csv_path
        self.extra_cols = extra_cols or {}
        if os.path.exists(csv_path):
            self.rows = pd.read_csv(csv_path).to_dict("records")
        else:
            self.rows = []

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch_num = int(round(state.epoch))
        if any(r.get("epoch") == epoch_num for r in self.rows):
            return control  # already evaluated this epoch (resume case)

        pred_log, valid_mask, gen_texts, gen_time = generate_predictions(model, self.eval_df)
        y_true_log = self.eval_df["target"].to_numpy(dtype=float)
        actual_usd = self.eval_df["target_usd"].to_numpy(dtype=float)
        pred_usd = np.clip(np.expm1(pred_log), a_min=0, a_max=None)

        metrics = evaluate_regression(y_true_log, pred_log, actual_usd, pred_usd)
        metrics["epoch"] = epoch_num
        metrics["Parse_Success_Rate"] = float(valid_mask.mean())
        metrics["Generation_Time_Seconds"] = gen_time
        metrics.update(self.extra_cols)

        self.rows.append(metrics)
        pd.DataFrame(self.rows).to_csv(self.csv_path, index=False)
        print(f"[epoch {epoch_num}] RMSE_log={metrics['RMSE_log']:.4f}  "
              f"R2_log={metrics['R2_log']:.4f}  parse_rate={metrics['Parse_Success_Rate']*100:.1f}%")
        return control


---
# Baseline training (fixed hyperparameters)

In [ ]:
cleanup_gpu()

model = make_model_with_lora(BASELINE_PARAMS)
tokenized_train = tokenize_split(train_df)

# Resume support: if BASELINE_OUTPUT_DIR already has Trainer checkpoints from a
# previous (interrupted) Kaggle session, pick up training from the last one.
last_checkpoint = None
if os.path.isdir(BASELINE_OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(BASELINE_OUTPUT_DIR)
    if last_checkpoint:
        print("Resuming from checkpoint:", last_checkpoint)

training_args = TrainingArguments(
    output_dir=BASELINE_OUTPUT_DIR,
    num_train_epochs=BASELINE_PARAMS["num_train_epochs"],
    per_device_train_batch_size=BASELINE_PARAMS["train_batch_size"],
    gradient_accumulation_steps=BASELINE_PARAMS["gradient_accumulation_steps"],
    learning_rate=BASELINE_PARAMS["learning_rate"],
    warmup_ratio=BASELINE_PARAMS["warmup_ratio"],
    weight_decay=BASELINE_PARAMS["weight_decay"],
    lr_scheduler_type=BASELINE_PARAMS["lr_scheduler_type"],
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="no",       # we run our own custom generative eval via the callback below
    bf16=USE_BF16,
    fp16=USE_FP16,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED,
    disable_tqdm=True,
)

# Per-epoch full 9-metric eval on val_df -> baseline_epoch_metrics.csv (resume-safe)
epoch_callback = EpochEvalCallback(eval_df=val_df, csv_path=EPOCH_METRICS_CSV)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    callbacks=[epoch_callback],
)

torch.cuda.empty_cache()
t0 = time.perf_counter()
trainer.train(resume_from_checkpoint=last_checkpoint)
training_time = time.perf_counter() - t0
print(f"Total training time: {training_time/60:.1f} min")


## Cell — Final evaluation on held-out test set

In [ ]:
# Final eval on the untouched test set
pred_log, valid_mask, gen_texts, gen_time = generate_predictions(trainer.model, test_df)

y_true_log = test_df["target"].to_numpy(dtype=float)
actual_usd = test_df["target_usd"].to_numpy(dtype=float)
pred_usd = np.clip(np.expm1(pred_log), a_min=0, a_max=None)

final_metrics = evaluate_regression(y_true_log, pred_log, actual_usd, pred_usd)
final_metrics["Training_Time_Seconds"] = training_time
final_metrics["Generation_Time_Seconds"] = gen_time
final_metrics["Parse_Success_Rate"] = float(valid_mask.mean())

baseline_final_row = {
    "Model": f"TinyLlama-1.1B-LoRA-Baseline-{SERIALIZATION}",
    "Serialization": SERIALIZATION,
    "Stage": "baseline",
    **BASELINE_PARAMS,
    **final_metrics,
}
baseline_final_df = pd.DataFrame([baseline_final_row])
display(baseline_final_df.round(6))
baseline_final_df.to_csv(FINAL_METRICS_CSV, index=False)

# Per-row test predictions, kept for XAI / error analysis later
baseline_pred_df = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "actual_log_target": test_df["target"].to_numpy(),
    "predicted_log_target": pred_log,
    "raw_generated_text": gen_texts,
    "numeric_parse_valid": valid_mask,
})
baseline_pred_df.to_csv("./baseline_test_predictions.csv", index=False)

print("Saved:", EPOCH_METRICS_CSV, FINAL_METRICS_CSV, "./baseline_test_predictions.csv")


## Cell — Save model + shared_config.json handoff file

In [ ]:
# Save adapter + tokenizer for downstream use (Optuna notebook input, later XAI work)
trainer.model.save_pretrained(BASELINE_MODEL_DIR)
tokenizer.save_pretrained(BASELINE_MODEL_DIR)

# ------------------------------------------------------------------------
# CRITICAL HANDOFF FILE: locks in the exact train/val id split + eval
# constants so the Optuna notebook evaluates on IDENTICAL data.
# ------------------------------------------------------------------------
shared_config = {
    "serialization": SERIALIZATION,
    "model_name": MODEL_NAME,
    "fallback_log_target": fallback_log_target,
    "max_reasonable_log_target": max_reasonable_log_target,
    "max_length": MAX_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "system_prompt": SYSTEM_PROMPT,
    "train_ids": train_df["id"].tolist(),
    "val_ids": val_df["id"].tolist(),
    "baseline_params": BASELINE_PARAMS,
}
with open(SHARED_CONFIG_PATH, "w") as f:
    json.dump(shared_config, f)

print("Saved shared_config.json with", len(shared_config["train_ids"]), "train ids and",
      len(shared_config["val_ids"]), "val ids")
print("Baseline model saved to:", BASELINE_MODEL_DIR)


---
# Notes — `tinyllama_baseline.ipynb`

- **Run once per serialization**: set `SERIALIZATION` at the top, then Save Version → Save & Run All.
- **Outputs produced** (all land in Kaggle's notebook output automatically):
  - `baseline_model/` — LoRA adapter + tokenizer (safetensors, HF-loadable)
  - `baseline_epoch_metrics.csv` — full 9-metric eval on `val_df` after every epoch
  - `baseline_final_metrics.csv` — full 9-metric eval on `test_df`, once, at the end
  - `baseline_test_predictions.csv` — per-row predictions + raw generated text (for error analysis / XAI)
  - `shared_config.json` — **required by `tinyllama_optuna.ipynb`**: locks the exact
    train/val id split and eval constants so Optuna trials are scored on identical data.
  - `{BASELINE_OUTPUT_DIR}/checkpoint-*` — Trainer checkpoints; if this notebook is
    interrupted and rerun, training resumes automatically from the last saved epoch.
- **To use in the Optuna notebook**: Save Version → Save & Run All on this notebook first,
  then in `tinyllama_optuna.ipynb` use "Add Input" → Notebooks tab → select this notebook.
  Its outputs mount read-only at `/kaggle/input/<this-notebook-slug>/...`.
